# CF Modeling — Visualization

Renders flatmap visualizations of CF model outputs using Hedger's vicsompy plotting code.

## Key design choices

**Fig 3a (2D colormap) uses raw split R².** Hedger's Fig 3a uses `frame['test_scores_somato_score']`
and `frame['test_scores_visual_score']` — these are the raw (un-null-corrected) split R² scores.
Raw R² has frac>0 ≈ 82–87%, giving the full smooth coverage seen in Hedger's figure.

**1D topographic maps use null-corrected R².** `R2_nc = R2_split − R2_null` removes the
global-signal component and reveals the genuine topographic gradient. This is useful for
showing spatial specificity but gives sparser coverage (frac>0 ≈ 47–62%).

**Integration score uses raw R².** The product map √(R²_A × R²_B) uses raw (not
null-corrected) split scores.

## Validation

Use `ROI_A='3b'`, `ROI_B='V1'` on correctly preprocessed group-average data to verify
the pipeline reproduces Hedger et al. Fig 3a.

**Note:** The group_average_sg_psc data is CORRUPTED (PSC+SG order bug). Rerun
`bash run_analysis.sh preprocess avg` to regenerate correct data before interpreting results.

## Pipeline prerequisites

```bash
bash cf_modeling/run_analysis.sh geometry
bash cf_modeling/run_analysis.sh avg
# OR for a single pair:
python cf_modeling/01_extract_geometry.py --roi-a 3b --roi-b V1 ...
python cf_modeling/02_fit_cf_model.py    --roi-a 3b --roi-b V1 ...
python cf_modeling/integration_maps.py  --roi-a 3b --roi-b V1 ...
```

In [5]:
%load_ext autoreload
%autoreload 2

import sys, os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import cortex

VICSOMPY_REPO = "/home/amin/Research/Representation/Movie/Vicarious_somatotopy"
if VICSOMPY_REPO not in sys.path:
    sys.path.insert(0, VICSOMPY_REPO)

from vicsompy.vis import Plot, basic_plot
from vicsompy.utils import load_plot_pkg_yaml

# Must point to hedger filestore BEFORE any cortex.db access
PYCORTEX_STORE = "/home/amin/Research/Representation/Movie/data/hedger2026"
new_db = cortex.database.Database(PYCORTEX_STORE)
cortex.db = new_db
cortex.database.default_filestore = new_db
for _mod_name, _mod in sys.modules.items():
    if _mod_name.startswith("cortex") and hasattr(_mod, "db"):
        _mod.db = new_db
cortex.options.config.set("basic", "filestore", PYCORTEX_STORE)
CX_SUB = list(cortex.db.subjects.keys())[0]
print(f"Pycortex subject: {CX_SUB}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Pycortex subject: hcp_999999_draw_NH


In [6]:
# =============================================================================
# CONFIG — edit before running
# =============================================================================
# Hedger Fig 3a validation: ROI_A='3b', ROI_B='V1', MODE='group_average'
# Our audiovisual analysis:  ROI_A='TA2', ROI_B='MST' (or A1/V1, A5/FFC, etc.)

ROI_A = "3b"            # seed A → RED channel in 2D map
ROI_B = "V1"            # seed B → BLUE channel in 2D map
MODE  = "group_average" # "group_average" | "per_subject"

OUT_BASE = "/home/amin/Research/Representation/Movie/outputs/cf_modeling"
TEMPLATE_CIFTI = (
    "/home/amin/Research/Representation/Movie/data/preprocessed/"
    "average_sub/sg_psc/group_average_sg_psc_cortex_59k.dtseries.nii"
)

roi_root = f"{OUT_BASE}/{MODE}/{ROI_A}_{ROI_B}"
prep_dir = f"{roi_root}/prep" if MODE == "group_average" else f"{roi_root}/group"
suffix   = "" if MODE == "group_average" else "_avg"
FIG_DIR  = f"{roi_root}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

print(f"ROI pair : {ROI_A} x {ROI_B}  ({MODE})")
print(f"prep dir : {prep_dir}")
print(f"fig dir  : {FIG_DIR}")

ROI pair : 3b x V1  (group_average)
prep dir : /home/amin/Research/Representation/Movie/outputs/cf_modeling/group_average/3b_V1/prep
fig dir  : /home/amin/Research/Representation/Movie/outputs/cf_modeling/group_average/3b_V1/figures


In [7]:
# =============================================================================
# Load R2 maps
# =============================================================================
# IMPORTANT: visualize with R2_nc (null-corrected), NOT raw R2.
# Raw R2 has a large global-signal component (every vertex gets high R2 from
# the seed mean timecourse), so the map looks patchy and featureless.
# R2_nc = R2_split - R2_null removes the trivial component, revealing
# the topographic gradient (what Hedger's frame['test_scores_*'] contains).

def _load(name):
    path = os.path.join(prep_dir, f"{name}{suffix}.npy")
    arr  = np.load(path)
    print(f"  {name+suffix:<30s}  shape={arr.shape}  "
          f"mean={arr.mean():+.4f}  frac>0={np.mean(arr>0):.1%}  max={arr.max():.4f}")
    return arr

print("Loading ...")
R2_a_nc = _load(f"R2_{ROI_A}_nc")  # null-corrected: use for topographic viz
R2_b_nc = _load(f"R2_{ROI_B}_nc")
R2_a    = _load(f"R2_{ROI_A}")     # raw split R2: used in product_map
R2_b    = _load(f"R2_{ROI_B}")
R2_full = _load("R2_full")
product = _load("product_map")     # sqrt(R2_A * R2_B), raw

_path_nc = os.path.join(prep_dir, f"product_map_nc{suffix}.npy")
if os.path.exists(_path_nc):
    product_nc = _load("product_map_nc")
else:
    product_nc = np.sqrt(np.clip(R2_a_nc, 0, None) * np.clip(R2_b_nc, 0, None)).astype(np.float32)
    print(f"  {'product_map_nc'+suffix:<30s}  shape={product_nc.shape}  "
          f"mean={product_nc.mean():+.4f}  frac>0={np.mean(product_nc>0):.1%}  "
          f"max={product_nc.max():.4f}  (computed inline)")

Loading ...
  R2_3b_nc                        shape=(108441,)  mean=+0.0075  frac>0=46.8%  max=0.9452
  R2_V1_nc                        shape=(108441,)  mean=+0.0376  frac>0=61.7%  max=0.7690
  R2_3b                           shape=(108441,)  mean=+0.0433  frac>0=81.6%  max=0.9630
  R2_V1                           shape=(108441,)  mean=+0.0853  frac>0=87.3%  max=0.9528
  R2_full                         shape=(108441,)  mean=+0.1286  frac>0=87.6%  max=0.9631
  product_map                     shape=(108441,)  mean=+0.0114  frac>0=31.8%  max=0.2299
  product_map_nc                  shape=(108441,)  mean=+0.0114  frac>0=31.8%  max=0.2299  (computed inline)


In [8]:
# =============================================================================
# Grayordinate -> full sphere (118584 vertices)
# =============================================================================
# HCP 59k_fs_LR: 59292 sphere vertices per hemisphere.
# 59412 grayordinates (medial wall excluded) -> (118584,) with zeros at wall.

bm_axis = nib.load(TEMPLATE_CIFTI).header.get_axis(1)

def to_sphere(arr):
    sphere = np.zeros(118584, dtype=np.float32)
    pos = 0
    for name, _, model in bm_axis.iter_structures():
        n = len(model.vertex)
        if "CORTEX_LEFT" in name:
            sphere[model.vertex] = arr[pos:pos+n]
        elif "CORTEX_RIGHT" in name:
            sphere[59292 + model.vertex] = arr[pos:pos+n]
        else:
            pos += n; continue
        pos += n
    return sphere

sph_full    = to_sphere(R2_full)
sph_a       = to_sphere(R2_a)       # raw split R² — for Fig 3a (matches Hedger)
sph_b       = to_sphere(R2_b)
sph_a_nc    = to_sphere(R2_a_nc)    # null-corrected — for 1D topographic maps
sph_b_nc    = to_sphere(R2_b_nc)
sph_prod    = to_sphere(product)
sph_prod_nc = to_sphere(product_nc)

# Fig 3a: raw R² clipped >= 0 (Hedger plots test_scores directly, no null correction)
sph_a_pos    = np.clip(sph_a,    0, None)
sph_b_pos    = np.clip(sph_b,    0, None)
# Null-corrected: sparse but spatially specific
sph_a_nc_pos = np.clip(sph_a_nc, 0, None)
sph_b_nc_pos = np.clip(sph_b_nc, 0, None)

print(f"Raw:  sph_a frac>0={np.mean(sph_a>0):.1%}  sph_b frac>0={np.mean(sph_b>0):.1%}")
print(f"NC:   sph_a frac>0={np.mean(sph_a_nc>0):.1%}  sph_b frac>0={np.mean(sph_b_nc>0):.1%}")

FileNotFoundError: No such file or no access: '/home/amin/Research/Representation/Movie/data/preprocessed/average_sub/sg_psc/group_average_sg_psc_cortex_59k.dtseries.nii'

In [ ]:
# Colourbar ceilings
# VMAX_RAW: for Fig 3a 2D plot (raw R²) — 95th pct of positive raw values, cap 0.20
# VMAX_NC:  for 1D null-corrected maps — 95th pct of positive nc values, cap 0.15
all_raw = np.concatenate([sph_a_pos[sph_a_pos > 0], sph_b_pos[sph_b_pos > 0]])
VMAX_RAW = min(float(np.percentile(all_raw, 95)), 0.20) if len(all_raw) > 0 else 0.10

all_nc = np.concatenate([sph_a_nc_pos[sph_a_nc_pos > 0], sph_b_nc_pos[sph_b_nc_pos > 0]])
VMAX_NC  = min(float(np.percentile(all_nc, 95)),  0.15) if len(all_nc) > 0 else 0.10

print(f"VMAX_RAW (Fig 3a):    {VMAX_RAW:.4f}")
print(f"VMAX_NC  (1D maps):   {VMAX_NC:.4f}")

In [ ]:
# =============================================================================
# Fig 3a -- 2D continuous colormap  (Hedger et al. 2025 style)
# =============================================================================
# Convention matching Hedger:
#   dat  = visual / ROI_B  -> BLUE channel   (test_scores_visual_score)
#   dat2 = somato / ROI_A  -> RED  channel   (test_scores_somato_score)
# Both axes: RAW split R² clipped to >= 0.
# Hedger uses raw scores directly (not null-corrected) for this figure.
#
# CRITICAL: dat2= must be passed to uber_plot(), NOT to Plot.__init__().

plotviews = load_plot_pkg_yaml()['plotviews']
cfg_2d = plotviews['variance_decomp_full'].copy()
cfg_2d.update({
    'vmax': VMAX_RAW, 'vmax2': VMAX_RAW,
    'vmin': 0.0,      'vmin2': 0.0,
    'with_colorbar': True,
    'with_rois': False,
    'subject': CX_SUB,
})
cfg_2d.pop('ROIS', None)

plotter_2d = Plot(**cfg_2d)
fig_2d = plotter_2d.uber_plot(dat=sph_b_pos, dat2=sph_a_pos)
fig_2d.suptitle(f"Fig 3a  {ROI_A} (red) x {ROI_B} (blue)  raw split R²",
                y=1.02, fontsize=11)
plt.tight_layout()
out = f"{FIG_DIR}/fig3a_{ROI_A}_{ROI_B}_2d_colormap.pdf"
fig_2d.savefig(out, bbox_inches='tight', dpi=200)
print(f"Saved: {out}")
plt.show()

In [ ]:
# =============================================================================
# 1D flatmaps -- individual null-corrected R2 per ROI
# =============================================================================

def _plot_1d(sph, roi, vmax, title):
    fig = basic_plot(np.where(sph > 0, sph, np.nan),
                     vmax=vmax, subject=CX_SUB, colorbar=True, rois=False)
    fig.suptitle(title, y=1.02, fontsize=11)
    out = f"{FIG_DIR}/R2_nc_{roi}.pdf"
    fig.savefig(out, bbox_inches='tight', dpi=200)
    print(f"Saved: {out}")
    plt.show()

_plot_1d(sph_a_nc, ROI_A, VMAX_NC, f"R2_nc  {ROI_A}")
_plot_1d(sph_b_nc, ROI_B, VMAX_NC, f"R2_nc  {ROI_B}")

In [ ]:
# =============================================================================
# Integration score -- sqrt(R2_A * R2_B)  [raw, not null-corrected]
# =============================================================================

pos_vals  = sph_prod[sph_prod > 0]
vmax_prod = min(float(np.percentile(pos_vals, 95)), 0.10) if len(pos_vals) > 0 else 0.05

fig_p = basic_plot(np.where(sph_prod > 0, sph_prod, np.nan),
                   vmax=vmax_prod, subject=CX_SUB, colorbar=True, rois=False)
fig_p.suptitle(f"Integration score  sqrt(R2_{ROI_A} x R2_{ROI_B})", y=1.02, fontsize=11)
out = f"{FIG_DIR}/integration_score_{ROI_A}_{ROI_B}.pdf"
fig_p.savefig(out, bbox_inches='tight', dpi=200)
print(f"Saved: {out}")
plt.show()

In [ ]:
# =============================================================================
# Null-corrected integration score -- sqrt(clip(R2_A_nc,0) * clip(R2_B_nc,0))
# =============================================================================
# Compared to the raw product_map, this shows bimodal regions that have
# *genuine topographic* responses from both ROIs (above the null-model baseline).

pos_nc    = sph_prod_nc[sph_prod_nc > 0]
vmax_prod_nc = min(float(np.percentile(pos_nc, 95)), 0.10) if len(pos_nc) > 0 else 0.05

fig_pnc = basic_plot(
    np.where(sph_prod_nc > 0, sph_prod_nc, np.nan),
    vmax=vmax_prod_nc, subject=CX_SUB, colorbar=True, rois=False,
)
fig_pnc.suptitle(
    f"Integration score (NC)  sqrt(clip(R2_{ROI_A}_nc,0) x clip(R2_{ROI_B}_nc,0))",
    y=1.02, fontsize=11,
)
out = f"{FIG_DIR}/integration_score_nc_{ROI_A}_{ROI_B}.pdf"
fig_pnc.savefig(out, bbox_inches='tight', dpi=200)
print(f"Saved: {out}")
plt.show()

In [ ]:
# =============================================================================
# 2D: modality balance x integration score
# =============================================================================
# dat  = modality_balance in [-1, +1]:  -1 -> ROI_B dominant, +1 -> ROI_A dominant
# dat2 = integration score (product_map)
# Shows WHERE integration happens and WHICH modality dominates.
# Uses null-corrected R² for balance (more specific) with raw product for integration.

modality_balance = (
    (R2_a_nc - R2_b_nc) / (np.abs(R2_a_nc) + np.abs(R2_b_nc) + 1e-8)
).astype(np.float32)
sph_balance = to_sphere(modality_balance)

pos_vals  = sph_prod[sph_prod > 0]
vmax_prod = min(float(np.percentile(pos_vals, 95)), 0.10) if len(pos_vals) > 0 else 0.05

cfg_int = plotviews['variance_decomp_full'].copy()
cfg_int.update({
    'vmax': 1.0, 'vmax2': vmax_prod,
    'vmin': -1.0, 'vmin2': 0.0,
    'with_colorbar': True,
    'with_rois': False,
    'subject': CX_SUB,
})
cfg_int.pop('ROIS', None)

plotter_int = Plot(**cfg_int)
fig_int = plotter_int.uber_plot(dat=sph_balance, dat2=sph_prod)
fig_int.suptitle("Modality balance (dat) x Integration score (dat2)", y=1.02, fontsize=11)
plt.tight_layout()
out = f"{FIG_DIR}/integration_balance_{ROI_A}_{ROI_B}.pdf"
fig_int.savefig(out, bbox_inches='tight', dpi=200)
print(f"Saved: {out}")
plt.show()

In [ ]:
# =============================================================================
# CIFTI outputs summary
# =============================================================================

cifti_dir = f"{roi_root}/cifti_maps"
print(f"CIFTI maps in: {cifti_dir}")
if os.path.isdir(cifti_dir):
    for fname in sorted(os.listdir(cifti_dir)):
        size = os.path.getsize(os.path.join(cifti_dir, fname)) / 1024
        print(f"  {fname:<55s}  {size:6.0f} KB")
else:
    print("  (not found -- run integration_maps.py first)")

In [ ]:
# =============================================================================
# Summary statistics table
# =============================================================================

rows = [
    (f"R2_{ROI_A}",    R2_a),
    (f"R2_{ROI_B}",    R2_b),
    (f"R2_{ROI_A}_nc", R2_a_nc),
    (f"R2_{ROI_B}_nc", R2_b_nc),
    ("R2_full",        R2_full),
    ("product_map",    product),
    ("product_map_nc", product_nc),
]
print(f"{'Map':<22}  {'Mean':>8}  {'Frac>0':>7}  {'Max':>8}  {'Min':>8}")
print("-" * 60)
for name, arr in rows:
    print(f"{name:<22}  {arr.mean():>8.4f}  {np.mean(arr>0):>7.1%}"
          f"  {arr.max():>8.4f}  {arr.min():>8.4f}")

In [ ]:
# =============================================================================
# Bimodal mask — vertices where BOTH R2_nc > 0
# =============================================================================
# Binary map: 1 = genuine bimodal response, 0 = unimodal or non-responsive.
# Corresponds to bimodal_mask.dscalar.nii saved by integration_maps.py.

bimodal_mask = ((R2_a_nc > 0) & (R2_b_nc > 0)).astype(np.float32)
sph_bimask   = to_sphere(bimodal_mask)

fig_bm = basic_plot(
    np.where(sph_bimask > 0, sph_bimask, np.nan),
    vmax=1.0, subject=CX_SUB, colorbar=True, rois=False,
)
fig_bm.suptitle(f"Bimodal mask  {ROI_A} x {ROI_B}  (R2_nc > 0 in both)",
                y=1.02, fontsize=11)
out = f"{FIG_DIR}/bimodal_mask_{ROI_A}_{ROI_B}.pdf"
fig_bm.savefig(out, bbox_inches='tight', dpi=200)
print(f"Saved: {out}")
plt.show()

In [ ]:
# =============================================================================
# R2_full — full banded-ridge model fit  (QC)
# =============================================================================
# Raw full-model R². High everywhere due to global signal; useful as a sanity
# check that the model fitted successfully. Corresponds to R2_full.dscalar.nii.

pos_full  = sph_full[sph_full > 0]
vmax_full = min(float(np.percentile(pos_full, 95)), 0.50) if len(pos_full) > 0 else 0.20

fig_full = basic_plot(
    np.where(sph_full > 0, sph_full, np.nan),
    vmax=vmax_full, subject=CX_SUB, colorbar=True, rois=False,
)
fig_full.suptitle("R2_full  (full banded-ridge model, QC)", y=1.02, fontsize=11)
out = f"{FIG_DIR}/R2_full_{ROI_A}_{ROI_B}.pdf"
fig_full.savefig(out, bbox_inches='tight', dpi=200)
print(f"Saved: {out}")
plt.show()